In [3]:
import numpy as np
from xgboost import XGBRegressor 
from sklearn.metrics import r2_score 
from data_processor import DataReader, DataPrep
from sklearn.model_selection import GridSearchCV
from cv_generator import train_val_split, ExpandingWindowCV
from sklearn.ensemble import RandomForestRegressor
from model_eval import *
import pandas as pd
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from functools import partial
from lightgbm import LGBMRegressor
import warnings


In [ ]:
# daily_data_path = r'data/daily_data'
# intraday_data_path = r'data/intraday_data'
# intraday_df = DataReader.read_intraday_data(intraday_data_path)
# daily_df = DataReader.read_daily_data(daily_data_path)
# daily_df.to_pickle('daily_df.pkl')
# intraday_df.to_pickle('intraday_df.pkl')

In [5]:
daily_df = pd.read_pickle('daily_df.pkl')
intraday_df =  pd.read_pickle('intraday_df.pkl')
data_prep = DataPrep(intraday_df, daily_df)
target_df = data_prep.get_target(clip_MAD=True, normalize= True)
X_df = data_prep.get_features()#indicators= {'RSI_14':partial(ta.rsi, length= 14), 'RSI_5':partial(ta.rsi, length= 5)})
input_data = X_df.join(target_df[['y', 'y_actual_clipped']], how = 'right')

### Train Val Split

In [14]:
features = ['CumReturnResid',  'Rolling_Return_5d', 'Rolling_Return_10d', 'Rolling_Return_20d','NYSE', 'IntradayRSI']#, 'RSI_14']#,'Stock_Split', 'Dividend', 'Rolling_Return_20d', 'EarlyClose', 'NextHoliday']
clipped_returns = [col for col in input_data.columns if 'clipped' in col and 'Return' in col]
features = [f'Rolling_Return_{i}d_clipped' for i in [5, 10]]  + ['CumReturnResid', 'IntradayRSI', 'NYSE']# 'VolumeChangeNormalize']#, 'NYSE'] + 
input_data.dropna(subset=features, inplace=True)
train_data, val_data = train_val_split(input_data, 0.8)
print(features)
x_train, y_train = train_data[features], train_data['y']
x_val, y_val = val_data[features], val_data['y_actual_clipped']#val_data['y']
train_weights = train_data.MDV_63_sqrt.to_numpy()
val_weights = val_data.MDV_63_sqrt.to_numpy()

['Rolling_Return_5d_clipped', 'Rolling_Return_10d_clipped', 'CumReturnResid', 'IntradayRSI', 'NYSE']


In [11]:
train_data[clipped_returns + ['y']].corr()['y'].sort_values()

CumReturnResid_clipped       -9.240184e-03
Rolling_Return_1d_clipped    -4.331477e-03
Rolling_Return_3d_clipped    -2.869687e-03
Rolling_Return_2d_clipped    -2.712822e-03
Rolling_Return_8d_clipped    -5.044472e-04
Rolling_Return_18d_clipped   -2.790443e-04
Rolling_Return_20d_clipped   -2.041969e-04
Rolling_Return_4d_clipped    -2.015070e-04
Rolling_Return_19d_clipped    9.845719e-07
Rolling_Return_17d_clipped    3.551084e-04
Rolling_Return_6d_clipped     4.021751e-04
Rolling_Return_5d_clipped     4.429807e-04
Rolling_Return_7d_clipped     5.449434e-04
Rolling_Return_9d_clipped     5.615897e-04
Rolling_Return_11d_clipped    8.222813e-04
Rolling_Return_12d_clipped    1.166753e-03
Rolling_Return_16d_clipped    1.286858e-03
Rolling_Return_15d_clipped    1.305621e-03
Rolling_Return_13d_clipped    1.384951e-03
Rolling_Return_10d_clipped    1.469783e-03
Rolling_Return_14d_clipped    1.665686e-03
y                             1.000000e+00
Name: y, dtype: float64

In [10]:
train_data[features + ['y']].corr()['y'].sort_values()


IntradayRSI                  -1.306414e-02
CumReturnResid_clipped       -9.240184e-03
CumReturnResid               -6.896120e-03
Rolling_Return_1d_clipped    -4.331477e-03
Rolling_Return_3d_clipped    -2.869687e-03
Rolling_Return_2d_clipped    -2.712822e-03
NYSE                         -7.778001e-04
Rolling_Return_8d_clipped    -5.044472e-04
Rolling_Return_18d_clipped   -2.790443e-04
Rolling_Return_20d_clipped   -2.041969e-04
Rolling_Return_4d_clipped    -2.015070e-04
Rolling_Return_19d_clipped    9.845719e-07
Rolling_Return_17d_clipped    3.551084e-04
Rolling_Return_6d_clipped     4.021751e-04
Rolling_Return_5d_clipped     4.429807e-04
Rolling_Return_7d_clipped     5.449434e-04
Rolling_Return_9d_clipped     5.615897e-04
Rolling_Return_11d_clipped    8.222813e-04
Rolling_Return_12d_clipped    1.166753e-03
Rolling_Return_16d_clipped    1.286858e-03
Rolling_Return_15d_clipped    1.305621e-03
Rolling_Return_13d_clipped    1.384951e-03
Rolling_Return_10d_clipped    1.469783e-03
Rolling_Ret

### Base random forest grid search

In [ ]:
rf = RandomForestRegressor()

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [2, 4, 6, 10],
    'min_samples_leaf': [2, 5, 10]
}

expand_wind_rf = ExpandingWindowCV(rf, rf_param_grid) 
expand_wind_rf.fit(x_train, y_train, train_weights)
print(expand_wind_rf.grid_search.best_params_)
model_rf = ModelEval(RandomForestRegressor(),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights,\
    grid_search= expand_wind_rf.grid_search)   
model_rf.to_pickle('xgboost_iter1')
print(model_rf.weighted_r2())
print(model_rf.feature_importance())
print(model_rf.feature_importance_MDA())

### Base xgboost Grid Search

In [ ]:
xgb = XGBRegressor()
xgb_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_weight': [1, 5, 10],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

expand_wind_xgb = ExpandingWindowCV(xgb, xgb_param_grid) 
expand_wind_xgb.fit(x_train, y_train, train_weights)
print(expand_wind_xgb.grid_search.best_params_)
# best_param = {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'reg_alpha': 1, 'reg_lambda': 2, 'subsample': 0.7}
model = ModelEval(XGBRegressor(),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, grid_search= expand_wind_xgb.grid_search)
# model.to_pickle('xgboost_base')
print(model.weighted_r2())
print(model.feature_importance())
print(model.feature_importance_MDA())

In [ ]:
val_data

### Base LightGBM Grid Search

In [15]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(verbosity = -1)

lgbm_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 150],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_samples': [1, 5, 10],
    'lambda_l2': [0.05, 0.1, 0.2, 0.3]
}

# expand_wind_lgbm = ExpandingWindowCV(lgbm, lgbm_param_grid) 
# expand_wind_lgbm.fit(x_train, y_train, train_weights)
# model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, expand_wind_lgbm.grid_search)
# model_lgbm.to_pickle('lgbm_iter2')
# model_lgbm = load_model('lgbm_iter1')
# print(model_lgbm.grid_cv.best_params_)
best_params = {'colsample_bytree': 1.0, 'lambda_l1': 0.3, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_samples': 5, 'n_estimators': 100, 'subsample': 0.7}
model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), \
    val_data.EST_VOL_preday.to_numpy(), val_weights,best_params= best_params)
print(model_lgbm.weighted_r2())
print(model_lgbm.feature_importance())
print(model_lgbm.feature_importance_MDA())

6.9549219539055684e-06
{'IntradayRSI': 420, 'CumReturnResid': 398, 'Rolling_Return_5d_clipped': 331, 'Rolling_Return_10d_clipped': 313, 'NYSE': 31}
{'IntradayRSI': 0.000655029038632852, 'CumReturnResid': 0.000360467956067545, 'Rolling_Return_10d_clipped': 0.0003178951927623445, 'Rolling_Return_5d_clipped': 0.00031081699168228707, 'NYSE': 3.088491667831613e-06}
